# Transformers — Sentiment Analysis & Text Summarisation

The architecture behind GPT, BERT, Claude, and every modern LLM.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Self-Attention Intuition

Every token attends to every other token:
```
Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
```
'Bank' in 'river bank' vs 'bank account' — context changes meaning.
Transformers capture this via attention over the full sequence.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
words = ['The','model','attended','to','every','word','context']
n = len(words)
np.random.seed(42)
attn = np.random.dirichlet(np.ones(n),size=n)
attn = (attn+attn.T)/2
plt.figure(figsize=(7,6))
plt.imshow(attn,cmap='Purples'); plt.colorbar()
plt.xticks(range(n),words,rotation=45,ha='right')
plt.yticks(range(n),words)
plt.title('Self-Attention Weights (simulated)'); plt.tight_layout(); plt.show()

## Sentiment Analysis (TF-IDF baseline)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
reviews = [
    "This movie was absolutely fantastic! Great acting.",
    "Terrible film. Complete waste of time. Very disappointed.",
    "An incredible journey. Moved me to tears. Must watch!",
    "Boring and predictable. Fell asleep halfway through.",
    "Brilliant performances. Gripping from start to finish.",
    "Awful script. Poor characters. Worst film I have seen.",
    "Heartwarming and beautiful. A modern masterpiece.",
    "Dreadful. Plot made no sense. Acting was wooden.",
    "Outstanding! Best film of the decade. Highly recommend.",
    "Disappointing. Had high hopes but execution was poor.",
]
labels = [1,0,1,0,1,0,1,0,1,0]
X_tr,X_te,y_tr,y_te = train_test_split(reviews,labels,test_size=0.3,random_state=42)
vec = TfidfVectorizer(ngram_range=(1,2))
clf = LogisticRegression().fit(vec.fit_transform(X_tr),y_tr)
print(classification_report(y_te,clf.predict(vec.transform(X_te)),target_names=['Neg','Pos']))

## HuggingFace Sentiment (Kaggle/Colab)

```python
from transformers import pipeline
sentiment = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')
results = sentiment(['This course is amazing!', 'I wasted my money.'])
for r in results:
    print(r)  # {'label': 'POSITIVE', 'score': 0.99}
```

## Extractive Summarisation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
def summarise(text, n=3):
    sents = [s.strip() for s in text.split('.') if len(s.strip())>20]
    if len(sents)<=n: return text
    mat = TfidfVectorizer(stop_words='english').fit_transform(sents)
    scores = mat.sum(axis=1).A1
    top = sorted(np.argsort(scores)[-n:])
    return '. '.join(sents[i] for i in top) + '.'
article = (
    "Machine learning enables systems to learn from data. "
    "Deep learning uses many layers to learn hierarchical representations. "
    "Transformers have revolutionised NLP since 2017. "
    "The attention mechanism allows models to focus on relevant parts. "
    "BERT and GPT are pre-trained models achieving state-of-the-art results. "
    "Transfer learning enables fine-tuning on specific tasks with limited data."
)
print("Original:", len(article.split()), "words")
summary = summarise(article, 2)
print("Summary: ", len(summary.split()), "words")
print(summary)

## BERT vs GPT vs T5

| Model | Type | Best for |
|---|---|---|
| BERT | Encoder | Classification, NER, Q&A |
| GPT | Decoder | Text generation |
| T5 | Encoder-Decoder | Translation, summarisation |
| DistilBERT | Encoder | Fast inference, mobile |
| RoBERTa | Encoder | Better BERT training |

---
**Your turn:** Change `n=2` in the summariser. Does removing one sentence hurt quality?

**HuggingFace:** huggingface.co/spaces — try 'summarization' demos live.